### 🔗 Links and Resources
| Method / Config | Description |
|-----------------|-------------|
| [broadcast()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.functions.broadcast.html) | Marks a DataFrame to be broadcast to all worker nodes during a join |
| [DataFrame.hint()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.hint.html) | Applies a hint to the DataFrame (e.g. `"broadcast"`) |
| [DataFrame.explain()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.explain.html) | Prints the physical plan — look for `BroadcastHashJoin` or `BroadcastExchange` |
| [SparkConf.get()](https://spark.apache.org/docs/3.5.3/api/python/reference/pyspark.sql/api/pyspark.sql.SparkSession.conf.html) | Reads a runtime Spark configuration value |
| `spark.sql.autoBroadcastJoinThreshold` | Max table size (bytes) Spark will auto-broadcast; set to `-1` to disable |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

### 📌 Creating Sample DataFrames

We use a **large** fact table (`orders`) and a **small** dimension table (`products`).
Broadcasting the small table avoids a shuffle of the large one.

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
from pyspark.sql import functions as F

# Large fact table: orders
orders_schema = StructType([
    StructField("order_id",   IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity",   IntegerType(), True),
])

orders_data = [
    (1001, 1, 2), (1002, 3, 1), (1003, 2, 5), (1004, 1, 3),
    (1005, 4, 2), (1006, 2, 1), (1007, 5, 4), (1008, 3, 2),
    (1009, 1, 1), (1010, 4, 3),
]

df_orders = spark.createDataFrame(orders_data, schema=orders_schema)

# Small dimension table: products
products_schema = StructType([
    StructField("product_id",   IntegerType(), True),
    StructField("product_name", StringType(),  True),
    StructField("unit_price",   DoubleType(),  True),
])

products_data = [
    (1, "Widget A", 9.99),
    (2, "Widget B", 14.99),
    (3, "Gadget C", 24.99),
    (4, "Gadget D", 4.99),
    (5, "Part E",   1.99),
]

df_products = spark.createDataFrame(products_data, schema=products_schema)

print("Orders:")
df_orders.show()
print("Products:")
df_products.show()

### 📌 Reading the Broadcast Threshold from Configuration

In [ ]:
# spark.sql.autoBroadcastJoinThreshold controls the size limit (in bytes)
# below which Spark automatically broadcasts a table during a join.
# Default is 10 MB (10485760 bytes). Set to -1 to disable auto-broadcast.
try:
    threshold = spark.conf.get("spark.sql.autoBroadcastJoinThreshold")
except Exception:
    threshold = "10485760"  # Default 10 MB in bytes
threshold_mb = int(threshold) / (1024 * 1024)

print(f"spark.sql.autoBroadcastJoinThreshold = {threshold} bytes ({threshold_mb:.1f} MB)")

### 📌 Explicit Broadcast Join Using broadcast()

In [ ]:
# Wrapping the small DataFrame in broadcast() tells Spark to send
# the entire table to every executor instead of performing a shuffle join.
df_broadcast_join = df_orders.join(
    F.broadcast(df_products),
    on="product_id",
    how="inner"
)

display(df_broadcast_join)

### 📌 Examining the Physical Plan

Look for **`BroadcastHashJoin`** and **`BroadcastExchange`** in the output — these confirm Spark chose a broadcast strategy.

In [ ]:
# extended=True prints Parsed / Analyzed / Optimized / Physical plans
df_broadcast_join.explain(extended=False)

### 📌 Deep Dive: `DataFrame.explain()` — All Modes

`explain()` prints the **query execution plan** Spark builds before running your job.
Understanding it is essential for diagnosing joins, shuffles, and performance bottlenecks.

---

#### Signature

```python
DataFrame.explain(extended=False, mode=None)
```

| Parameter | Type | Default | Notes |
|-----------|------|---------|-------|
| `extended` | `bool` | `False` | Legacy flag — superseded by `mode` in Spark 3.0 |
| `mode` | `str` | `None` | Preferred way to choose plan detail level (Spark 3.0+) |

When `mode` is supplied it takes precedence over `extended`.

---

#### Available Modes

| Mode | Equivalent old call | What you see |
|------|---------------------|--------------|
| `"simple"` | `explain()` / `explain(False)` | **Physical plan only** — the execution tree Spark will actually run |
| `"extended"` | `explain(True)` | **All 4 plan stages**: Parsed → Analyzed → Optimized Logical → Physical |
| `"codegen"` | *(new in Spark 3.0)* | **Generated Java bytecode** for each WholeStageCodegen stage |
| `"cost"` | *(new in Spark 3.0)* | Physical plan **annotated with row/size statistics** used by the Cost-Based Optimizer |
| `"formatted"` | *(new in Spark 3.0)* | **Two-section output**: clean indented plan + per-node detail below — easiest to read |

---

#### The 4 Plan Stages (`extended` / `"extended"` mode)

```
Unresolved Logical Plan   ← what you wrote (column names not yet validated)
        ↓  Analyzer (resolves names, types, catalog lookups)
Resolved Logical Plan
        ↓  Optimizer (predicate pushdown, constant folding, join reordering …)
Optimized Logical Plan
        ↓  Planner (picks physical operators: HashJoin vs BroadcastHashJoin, etc.)
Physical Plan             ← what actually executes on the cluster
```

---

#### What to look for in a Broadcast Join plan

| Node | Meaning |
|------|---------|
| `BroadcastHashJoin` / `PhotonBroadcastHashJoin` | Spark chose broadcast strategy ✅ |
| `BroadcastExchange` | The small table is being shipped to all executors |
| `SortMergeJoin` | Shuffle join — both sides sorted and merged (expensive) |
| `ShuffledHashJoin` | Shuffle join — one side hashed (no sort, but memory-heavy) |
| `AdaptiveSparkPlan` | AQE is active; plan may change at runtime |

In [ ]:
# ── explain() mode showcase ──────────────────────────────────────────────────
# We use df_broadcast_join created above (orders JOIN broadcast(products)).
# Each block is separated by a banner so you can compare outputs side-by-side.

SEP = "\n" + "=" * 70 + "\n"

# ── 1. simple (default) ──────────────────────────────────────────────────────
# Physical plan only. Good for a quick sanity-check on join strategy.
print(SEP + "MODE: simple  (physical plan only)" + SEP)
df_broadcast_join.explain(mode="simple")

# ── 2. extended ──────────────────────────────────────────────────────────────
# Shows all 4 plan stages:
#   • Parsed Logical Plan   – raw AST before name/type resolution
#   • Analyzed Logical Plan – columns resolved against the catalog
#   • Optimized Logical Plan – after Catalyst rule-based optimizations
#   • Physical Plan          – the actual execution plan
print(SEP + "MODE: extended  (all 4 plan stages)" + SEP)
df_broadcast_join.explain(mode="extended")

# ── 3. formatted ─────────────────────────────────────────────────────────────
# Two-section layout:
#   Section 1: clean indented tree (like "simple" but better spaced)
#   Section 2: per-node detail block with arguments, output, statistics
# This is the most readable mode for everyday debugging (Spark 3.0+).
print(SEP + "MODE: formatted  (indented tree + per-node detail)" + SEP)
df_broadcast_join.explain(mode="formatted")

# ── 4. cost ───────────────────────────────────────────────────────────────────
# Physical plan annotated with row counts and byte sizes that the
# Cost-Based Optimizer used when choosing the join strategy.
# Useful for understanding WHY Spark picked broadcast vs sort-merge.
print(SEP + "MODE: cost  (physical plan + CBO statistics)" + SEP)
df_broadcast_join.explain(mode="cost")

# ── 5. codegen ────────────────────────────────────────────────────────────────
# Dumps the generated Java bytecode for each WholeStageCodegen stage.
# Rarely needed in day-to-day work; useful for Spark engine contributors
# or diagnosing codegen-specific bugs.
# NOTE: On Databricks with Photon enabled this section may be empty or
#       minimal because Photon uses native C++ execution, not JVM codegen.
print(SEP + "MODE: codegen  (generated Java bytecode per stage)" + SEP)
df_broadcast_join.explain(mode="codegen")

### 📌 Broadcast Join Using hint()

In [ ]:
# hint("broadcast") is equivalent to broadcast() but uses the DataFrame API hint mechanism.
# Useful when you cannot wrap the DataFrame in a function call directly.
df_hint_join = df_orders.join(
    df_products.hint("broadcast"),
    on="product_id",
    how="inner"
)

display(df_hint_join)
df_hint_join.explain(extended=False)

### 📌 Disabling Auto-Broadcast and Comparing Plans

In [ ]:
# Temporarily disable auto-broadcast to see the shuffle join plan
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

df_shuffle_join = df_orders.join(df_products, on="product_id", how="inner")
print("--- Plan WITHOUT broadcast (auto-broadcast disabled) ---")
df_shuffle_join.explain(extended=False)

# Restore the original threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", threshold)
print(f"Threshold restored to {threshold} bytes")